In [2]:
!conda install -c conda-forge libgfortran5 liblapack libblas -y

Error while loading conda entry point: conda-libmamba-solver (dlopen(/opt/anaconda3/lib/python3.12/site-packages/libmambapy/bindings.cpython-312-darwin.so, 0x0002): Library not loaded: @rpath/libarchive.20.dylib
  Referenced from: <496442DC-0EDE-3705-A2B5-401A4FC0D733> /opt/anaconda3/lib/libmamba.2.0.0.dylib
  Reason: tried: '/opt/anaconda3/lib/libarchive.20.dylib' (no such file), '/opt/anaconda3/lib/python3.12/site-packages/libmambapy/../../../libarchive.20.dylib' (no such file), '/opt/anaconda3/lib/python3.12/site-packages/libmambapy/../../../libarchive.20.dylib' (no such file), '/opt/anaconda3/bin/../lib/libarchive.20.dylib' (no such file), '/opt/anaconda3/bin/../lib/libarchive.20.dylib' (no such file), '/usr/local/lib/libarchive.20.dylib' (no such file), '/usr/lib/libarchive.20.dylib' (no such file, not in dyld cache))
Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 24.11.3
  latest version: 26.7.0

Please update conda by running

  

In [3]:
!conda install -c conda-forge numpy scipy scikit-learn -y

Error while loading conda entry point: conda-libmamba-solver (dlopen(/opt/anaconda3/lib/python3.12/site-packages/libmambapy/bindings.cpython-312-darwin.so, 0x0002): Library not loaded: @rpath/libarchive.20.dylib
  Referenced from: <496442DC-0EDE-3705-A2B5-401A4FC0D733> /opt/anaconda3/lib/libmamba.2.0.0.dylib
  Reason: tried: '/opt/anaconda3/lib/libarchive.20.dylib' (no such file), '/opt/anaconda3/lib/python3.12/site-packages/libmambapy/../../../libarchive.20.dylib' (no such file), '/opt/anaconda3/lib/python3.12/site-packages/libmambapy/../../../libarchive.20.dylib' (no such file), '/opt/anaconda3/bin/../lib/libarchive.20.dylib' (no such file), '/opt/anaconda3/bin/../lib/libarchive.20.dylib' (no such file), '/usr/local/lib/libarchive.20.dylib' (no such file), '/usr/lib/libarchive.20.dylib' (no such file, not in dyld cache))
Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 24.11.3
  latest version: 26.7.0

Please update conda by running

  

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import random

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

ImportError: dlopen(/opt/anaconda3/lib/python3.12/site-packages/scipy/linalg/_fblas.cpython-312-darwin.so, 0x0002): Library not loaded: @rpath/liblapack.3.dylib
  Referenced from: <3CDC9333-082B-3310-B2F6-E76203C017E5> /opt/anaconda3/lib/python3.12/site-packages/scipy/linalg/_fblas.cpython-312-darwin.so
  Reason: tried: '/opt/anaconda3/lib/python3.12/site-packages/scipy/linalg/../../../../liblapack.3.dylib' (no such file), '/opt/anaconda3/lib/python3.12/site-packages/scipy/linalg/../../../../liblapack.3.dylib' (no such file), '/opt/anaconda3/bin/../lib/liblapack.3.dylib' (no such file), '/opt/anaconda3/bin/../lib/liblapack.3.dylib' (no such file), '/usr/local/lib/liblapack.3.dylib' (no such file), '/usr/lib/liblapack.3.dylib' (no such file, not in dyld cache)

In [ ]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed:", SEED)

In [ ]:
dataset = pd.read_excel(
    "Expense_Dataset_With_NextMonthExpense.xlsx"
)

print("Dataset Shape:", dataset.shape)

display(dataset.head())

In [ ]:
dataset.info()

In [ ]:
print("\nMissing Values:")
print(dataset.isnull().sum())

In [ ]:
print(
    "\nDuplicate Rows:",
    dataset.duplicated().sum()
)

In [ ]:
dataset = dataset.drop_duplicates()

dataset = dataset.reset_index(drop=True)

print(
    "Dataset Shape After Removing Duplicates:",
    dataset.shape
)

In [ ]:
dataset = dataset.sort_values(
    by=[
        "User_ID",
        "Year",
        "Month"
    ]
).reset_index(drop=True)

In [ ]:
print(
    dataset["Month"].unique()
)

In [ ]:
month_order = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

if dataset["Month"].dtype == "object":

    dataset["Month"] = dataset[
        "Month"
    ].map(month_order)

In [ ]:
print(
    dataset["Month"].unique()
)

In [ ]:
features = [

    "Year",

    "Month",

    "Monthly_Income",

    "Bills",

    "Education",

    "Entertainment",

    "Food",

    "Health",

    "Shopping",

    "Transport",

    "Travel",

    "Total_Expense",

    "Savings",

    "Savings_Rate",

    "Expense_Ratio",

    "Average_Daily_Expense",

    "Weekend_Spending_Percentage",

    "Transaction_Count",

    "Spending_Profile"

]

target = "NextMonthExpense"

In [ ]:
missing_features = [
    column
    for column in features + [target]
    if column not in dataset.columns
]

print(
    "Missing columns:",
    missing_features
)

In [ ]:
print(
    dataset[target].describe()
)

In [ ]:
SEQUENCE_LENGTH = 6

In [ ]:
def create_sequences(
    data,
    features,
    target,
    sequence_length
):

    X = []
    y = []

    sequence_users = []

    for user_id, user_data in data.groupby(
        "User_ID"
    ):

        user_data = user_data.sort_values(
            by=["Year", "Month"]
        ).reset_index(drop=True)

        feature_values = user_data[
            features
        ].values

        target_values = user_data[
            target
        ].values

        for i in range(
            sequence_length,
            len(user_data)
        ):

            X.append(
                feature_values[
                    i-sequence_length:i
                ]
            )

            y.append(
                target_values[i]
            )

            sequence_users.append(
                user_id
            )

    return (
        np.array(X),
        np.array(y),
        np.array(sequence_users)
    )

In [ ]:
X, y, sequence_users = create_sequences(

    dataset,

    features,

    target,

    SEQUENCE_LENGTH

)

print(
    "X Shape:",
    X.shape
)

print(
    "y Shape:",
    y.shape
)

print(
    "Number of sequences:",
    len(X)
)

split_index = int(
    len(X) * 0.80
)

X_train = X[
    :split_index
]

X_test = X[
    split_index:
]

y_train = y[
    :split_index
]

y_test = y[
    split_index:
]

users_train = sequence_users[
    :split_index
]

users_test = sequence_users[
    split_index:
]

In [ ]:
print(
    "Training:",
    X_train.shape
)

print(
    "Testing:",
    X_test.shape
)

In [ ]:
num_features = X_train.shape[2]

scaler = StandardScaler()

X_train_2d = X_train.reshape(
    -1,
    num_features
)

X_test_2d = X_test.reshape(
    -1,
    num_features
)

scaler.fit(
    X_train_2d
)

X_train_scaled = scaler.transform(
    X_train_2d
)

X_test_scaled = scaler.transform(
    X_test_2d
)

X_train = X_train_scaled.reshape(
    X_train.shape
)

X_test = X_test_scaled.reshape(
    X_test.shape
)

print(
    "Scaled X_train:",
    X_train.shape
)

print(
    "Scaled X_test:",
    X_test.shape
)

validation_index = int(
    len(X_train) * 0.85
)

X_train_final = X_train[
    :validation_index
]

X_val = X_train[
    validation_index:
]

y_train_final = y_train[
    :validation_index
]

y_val = y_train[
    validation_index:
]

print(
    "Training:",
    X_train_final.shape
)

print(
    "Validation:",
    X_val.shape
)

print(
    "Testing:",
    X_test.shape
)

In [ ]:
def build_lstm_model(
    units1=128,
    units2=64,
    dropout_rate=0.2,
    learning_rate=0.001
):

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        LSTM(
            units1,
            return_sequences=True
        )
    )

    model.add(
        Dropout(
            dropout_rate
        )
    )

    model.add(
        LSTM(
            units2,
            return_sequences=False
        )
    )

    model.add(
        Dropout(
            dropout_rate
        )
    )

    model.add(
        Dense(
            32,
            activation="relu"
        )
    )

    model.add(
        Dense(1)
    )

    optimizer = Adam(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
configurations = [

    {
        "name": "LSTM_64_32",

        "units1": 64,

        "units2": 32,

        "dropout": 0.20,

        "learning_rate": 0.001,

        "batch_size": 64
    },

    {
        "name": "LSTM_128_64",

        "units1": 128,

        "units2": 64,

        "dropout": 0.20,

        "learning_rate": 0.001,

        "batch_size": 64
    },

    {
        "name": "LSTM_128_64_D03",

        "units1": 128,

        "units2": 64,

        "dropout": 0.30,

        "learning_rate": 0.001,

        "batch_size": 64
    },

    {
        "name": "LSTM_256_128",

        "units1": 256,

        "units2": 128,

        "dropout": 0.20,

        "learning_rate": 0.001,

        "batch_size": 64
    },

    {
        "name": "LSTM_128_64_LR0005",

        "units1": 128,

        "units2": 64,

        "dropout": 0.20,

        "learning_rate": 0.0005,

        "batch_size": 64
    },

    {
        "name": "LSTM_256_128_D03",

        "units1": 256,

        "units2": 128,

        "dropout": 0.30,

        "learning_rate": 0.0005,

        "batch_size": 64
    },

    {
        "name": "LSTM_256_128_LR0001",

        "units1": 256,

        "units2": 128,

        "dropout": 0.20,

        "learning_rate": 0.0001,

        "batch_size": 64
    }

]

In [ ]:
tuning_results = []

trained_lstm_models = {}

training_histories = {}

for config in configurations:

    print("=" * 70)

    print(
        "Training:",
        config["name"]
    )

    model = build_lstm_model(

        units1=config["units1"],

        units2=config["units2"],

        dropout_rate=config["dropout"],

        learning_rate=config[
            "learning_rate"
        ]

    )

    early_stopping = EarlyStopping(

        monitor="val_loss",

        patience=10,

        restore_best_weights=True

    )

    reduce_lr = ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=5,

        min_lr=0.000001,

        verbose=0

    )

    history = model.fit(

        X_train_final,

        y_train_final,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=100,

        batch_size=config[
            "batch_size"
        ],

        callbacks=[
            early_stopping,
            reduce_lr
        ],

        verbose=0

    )

    val_predictions = model.predict(
        X_val,
        verbose=0
    ).flatten()

    val_mae = mean_absolute_error(
        y_val,
        val_predictions
    )

    val_rmse = np.sqrt(
        mean_squared_error(
            y_val,
            val_predictions
        )
    )

    val_r2 = r2_score(
        y_val,
        val_predictions
    )

    val_mape = (
        mean_absolute_percentage_error(
            y_val,
            val_predictions
        ) * 100
    )

    tuning_results.append({

        "Model": config["name"],

        "MAE": val_mae,

        "RMSE": val_rmse,

        "R²": val_r2,

        "MAPE (%)": val_mape,

        "Epochs": len(
            history.history["loss"]
        )

    })

    trained_lstm_models[
        config["name"]
    ] = model

    training_histories[
        config["name"]
    ] = history

    print(
        "Validation MAE:",
        round(val_mae, 2)
    )

    print(
        "Validation RMSE:",
        round(val_rmse, 2)
    )

    print(
        "Validation R²:",
        round(val_r2, 4)
    )

    print(
        "Validation MAPE:",
        round(val_mape, 2),
        "%"
    )

In [ ]:
lstm_tuning_results = pd.DataFrame(
    tuning_results
)

lstm_tuning_results = (
    lstm_tuning_results
    .sort_values(
        by="R²",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    lstm_tuning_results
)

In [ ]:
lstm_tuning_results.to_csv(
    "lstm_tuning_results.csv",
    index=False
)

In [ ]:
best_lstm_name = (
    lstm_tuning_results
    .iloc[0]["Model"]
)

best_lstm_model = (
    trained_lstm_models[
        best_lstm_name
    ]
)

best_history = (
    training_histories[
        best_lstm_name
    ]
)

print(
    "Best LSTM:",
    best_lstm_name
)

In [ ]:
final_predictions = (
    best_lstm_model
    .predict(
        X_test,
        verbose=0
    )
    .flatten()
)

In [ ]:
final_mae = mean_absolute_error(
    y_test,
    final_predictions
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        final_predictions
    )
)

final_r2 = r2_score(
    y_test,
    final_predictions
)

final_mape = (
    mean_absolute_percentage_error(
        y_test,
        final_predictions
    ) * 100
)

print("=" * 60)

print(
    "FINAL TUNED LSTM RESULTS"
)

print("=" * 60)

print(
    "MAE  :",
    round(final_mae, 2)
)

print(
    "RMSE :",
    round(final_rmse, 2)
)

print(
    "R²   :",
    round(final_r2, 4)
)

print(
    "MAPE :",
    round(final_mape, 2),
    "%"
)

In [ ]:
baseline_lstm = {

    "Model": "Baseline LSTM",

    "MAE": 6084.47,

    "RMSE": 8552.01,

    "R²": 0.978,

    "MAPE (%)": 5.57

}

tuned_lstm_result = {

    "Model": "Tuned LSTM",

    "MAE": final_mae,

    "RMSE": final_rmse,

    "R²": final_r2,

    "MAPE (%)": final_mape
}

lstm_comparison = pd.DataFrame([
    baseline_lstm,
    tuned_lstm_result
])

display(
    lstm_comparison
)

In [ ]:
plt.figure(
    figsize=(10, 5)
)

plt.plot(
    best_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    best_history.history[
        "val_loss"
    ],
    label="Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("MSE Loss")

plt.title(
    "LSTM Training and Validation Loss"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "lstm_training_validation_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(
    figsize=(7, 7)
)

plt.scatter(
    y_test,
    final_predictions,
    alpha=0.5
)

plt.plot(
    [
        y_test.min(),
        y_test.max()
    ],
    [
        y_test.min(),
        y_test.max()
    ],
    linestyle="--",
    linewidth=2
)

plt.xlabel(
    "Actual Expense (LKR)"
)

plt.ylabel(
    "Predicted Expense (LKR)"
)

plt.title(
    "LSTM Actual vs Predicted Expenses"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "lstm_actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
prediction_comparison = pd.DataFrame({

    "Actual": y_test,

    "Predicted": final_predictions

}).reset_index(drop=True)

prediction_comparison = (
    prediction_comparison
    .iloc[:100]
)

plt.figure(
    figsize=(15, 6)
)

plt.plot(
    prediction_comparison[
        "Actual"
    ],
    label="Actual"
)

plt.plot(
    prediction_comparison[
        "Predicted"
    ],
    label="Predicted"
)

plt.xlabel(
    "Test Sample"
)

plt.ylabel(
    "Expense (LKR)"
)

plt.title(
    "LSTM Actual vs Predicted Expenses - First 100 Test Samples"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "lstm_prediction_line_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
residuals = (
    y_test -
    final_predictions
)

plt.figure(
    figsize=(8, 5)
)

plt.scatter(
    final_predictions,
    residuals,
    alpha=0.5
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=2
)

plt.xlabel(
    "Predicted Expense (LKR)"
)

plt.ylabel(
    "Residual Error"
)

plt.title(
    "LSTM Residual Error Plot"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "lstm_residual_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
best_lstm_model.save(
    "expense_prediction_lstm_tuned.keras"
)

print(
    "LSTM model saved successfully!"
)

In [ ]:
joblib.dump(
    scaler,
    "lstm_feature_scaler.pkl"
)

print(
    "Scaler saved successfully!"
)

In [ ]:
joblib.dump(
    features,
    "lstm_feature_names.pkl"
)

print(
    "Feature names saved successfully!"
)

In [ ]:
joblib.dump(
    SEQUENCE_LENGTH,
    "lstm_sequence_length.pkl"
)

In [ ]:
final_result = pd.DataFrame({

    "Model": [
        "Tuned LSTM"
    ],

    "MAE": [
        final_mae
    ],

    "RMSE": [
        final_rmse
    ],

    "R²": [
        final_r2
    ],

    "MAPE (%)": [
        final_mape
    ]

})

display(
    final_result
)

final_result.to_csv(
    "tuned_lstm_results.csv",
    index=False
)

In [ ]:
prediction_results = pd.DataFrame({

    "Actual_Expense": y_test,

    "Predicted_Expense": final_predictions,

    "Residual": (
        y_test -
        final_predictions
    )

})

prediction_results.to_csv(
    "lstm_prediction_results.csv",
    index=False
)

display(
    prediction_results.head(20)
)

In [ ]:
mae_improvement = (
    (
        6084.47 -
        final_mae
    )
    /
    6084.47
) * 100

rmse_improvement = (
    (
        8552.01 -
        final_rmse
    )
    /
    8552.01
) * 100

r2_improvement = (
    final_r2 -
    0.978
)

mape_improvement = (
    (
        5.57 -
        final_mape
    )
    /
    5.57
) * 100

print(
    "MAE improvement:",
    round(mae_improvement, 2),
    "%"
)

print(
    "RMSE improvement:",
    round(rmse_improvement, 2),
    "%"
)

print(
    "R² improvement:",
    round(r2_improvement, 4)
)

print(
    "MAPE improvement:",
    round(mape_improvement, 2),
    "%"
)